# Team Statistical Leaders

- Batting Leaders
    - Batting Average
    - Home Runs (HR)
    - Runs Batted In (RBI)
    - Hits
    - Stolen Bases
- Pitching Leaders
    - Earned Run Average (ERA)
    - Strikeouts

##Batting Leaders

###Batting Average

In [0]:
%sql
SELECT
    team_name,
    ROUND((SUM(hits) / SUM(stat_at_bats)), 3) AS batting_avg
FROM gold.fact_batting_stats
GROUP BY team_name
ORDER BY batting_avg DESC
LIMIT (10)

###Hits

In [0]:
%sql
SELECT
    team_name,
    SUM(hits) AS tot_hits
FROM gold.fact_batting_stats
GROUP BY team_name
ORDER BY tot_hits DESC
LIMIT (10)

###Home runs

In [0]:
%sql
SELECT
    team_name,
    SUM(home_run) AS tot_hr
FROM gold.fact_batting_stats
GROUP BY team_name
ORDER BY tot_hr DESC
LIMIT (10)

###Runs batted in (RBI)

In [0]:
%sql
SELECT
    team_name,
    SUM(rbi) AS tot_rbi
FROM gold.fact_batting_stats
GROUP BY team_name
ORDER BY tot_rbi DESC
LIMIT (10)

###Stolen Bases

In [0]:
%sql
SELECT
    team_name,
    SUM(stolen_bases) AS tot_stolen_bases
FROM gold.fact_batting_stats
GROUP BY team_name
ORDER BY tot_stolen_bases DESC
LIMIT (10)

##Pitching Leaders

###Earned Run Average (ERA)

In [0]:
%sql
SELECT
    team_name,
    ROUND((SUM(earned_runs)*9) / SUM(innings_pitched_decimal),2) AS era
FROM gold.fact_pitching_stats
GROUP BY team_name
ORDER BY era
LIMIT 10 OFFSET 1

###Strikeouts

In [0]:
%sql
SELECT
    team_name,
    SUM(strikeouts_pitched) AS tot_strikeouts
FROM gold.fact_pitching_stats
GROUP BY team_name
ORDER BY tot_strikeouts DESC
LIMIT 10

# Player Statistical Leaders (Top 10)

- Batting Leaders
    - Batting Average
    - Home Runs (HR)
    - Runs Batted In (RBI)
    - Hits
    - Stolen Bases
- Pitching Leaders
    - Earned Run Average (ERA)
    - Strikeouts

##Batting Leaders

In [0]:
%sql
-- Everything in one
SELECT player_name, b.team_name,
       SUM(hits) AS hits, SUM(home_run) AS hr, SUM(rbi) AS rbi,
       SUM(stat_at_bats) AS ab,
       ROUND(SUM(hits) / NULLIF(SUM(stat_at_bats), 0), 3) AS batting_avg
FROM gold.fact_batting_stats b
JOIN gold.dim_teams t ON b.team_id = t.team_id
GROUP BY player_name, b.team_name
LIMIT 10

###Batting Average

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW player_ba AS
SELECT
    player_id,
    player_name,
    COUNT(game_pk) AS tot_games,
    SUM(stat_at_bats) AS tot_at_bats,
    SUM(walks),
    ROUND((SUM(stat_at_bats)+SUM(walks)) / NULLIF(COUNT(game_pk),0), 1) AS pa_per_game,
    ROUND(SUM(hits) / NULLIF(SUM(stat_at_bats),0), 3) AS batting_avg
FROM gold.fact_batting_stats
GROUP BY player_id, player_name
HAVING pa_per_game > 3.1 AND tot_games > 90
ORDER BY batting_avg DESC;
    


SELECT 
    player_id,
    player_name,
    batting_avg
FROM player_ba
LIMIT(10)

###Home Runs

In [0]:
%sql
SELECT
    player_id,
    player_name,
    SUM(home_run) AS tot_hr
FROM gold.fact_batting_stats
GROUP BY player_id, player_name
ORDER BY tot_hr DESC
LIMIT(10)

###Runs Batted In (RBI)

In [0]:
%sql
SELECT
    player_id,
    player_name,
    SUM(rbi) AS tot_rbi
FROM gold.fact_batting_stats
GROUP BY player_id, player_name
ORDER BY tot_rbi DESC
LIMIT(10)

###Hits

In [0]:
%sql
SELECT
    player_id,
    player_name,
    SUM(hits) AS tot_hits
FROM gold.fact_batting_stats
GROUP BY player_id, player_name
ORDER BY tot_hits DESC
LIMIT(10)

###Stolen Bases

In [0]:
%sql
SELECT
    player_id,
    player_name,
    SUM(stolen_bases) AS tot_stolen_bases
FROM gold.fact_batting_stats
GROUP BY player_id, player_name
ORDER BY tot_stolen_bases DESC
LIMIT(10)

###Pitching Leaders

###Earned Run Average (ERA)

In [0]:
%sql
SELECT
    player_id,
    player_name,
    ROUND((SUM(earned_runs)*9) / NULLIF(SUM(innings_pitched_decimal),0), 2) AS era
FROM gold.fact_pitching_stats
GROUP BY player_id, player_name
HAVING SUM(innings_pitched_decimal) > 121
ORDER BY era
LIMIT 10

###Strikeouts

In [0]:
%sql
SELECT
    player_id,
    player_name,
    SUM(strikeouts_pitched) AS tot_strikeouts
FROM gold.fact_pitching_stats
GROUP BY player_id, player_name
ORDER BY tot_strikeouts DESC
LIMIT 10

# Hot/cold streak - rolling performance trend

##Player rolling batting avg (7 games)

In [0]:
%sql
-- player's rolling batting avg and hits over 7 games
SELECT player_name, b.game_date,
       ROUND(AVG(hits*1.0/NULLIF(stat_at_bats,0)) OVER (
         PARTITION BY player_id ORDER BY b.game_date
         ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
       ), 3) AS rolling_7day_avg
FROM gold.fact_batting_stats b
JOIN gold.dim_games g ON b.game_pk = g.game_pk
WHERE player_name = 'Freddie Freeman'
ORDER BY game_date DESC

##Team rolling batting avg (7 games)

In [0]:
%sql
-- team's rolling batting avg over 7 games
WITH team_game_stats AS (
  SELECT
    b.team_id,
    t.team_name,
    b.game_date,
    SUM(b.hits) AS team_hits,
    SUM(b.stat_at_bats) AS team_at_bats
  FROM gold.fact_batting_stats b
  JOIN gold.dim_teams t ON b.team_id = t.team_id
  GROUP BY b.team_id, t.team_name, b.game_date
)

SELECT
  team_id, team_name, game_date,
  ROUND(SUM(team_hits) OVER (
    PARTITION BY team_id ORDER BY game_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) / 
    NULLIF(
    SUM(team_at_bats) OVER (
    PARTITION BY team_id ORDER BY game_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 0), 3) AS rolling_7game_avg
FROM team_game_stats
WHERE team_name = 'Milwaukee Brewers'
ORDER BY game_date DESC

##Teamns MOST RECENT Rolling batting avg (7 games)

In [0]:
%sql
--team's batting avg from last 7 games
WITH team_game_stats AS (
  SELECT
    b.team_id,
    t.team_name,
    b.game_date,
    SUM(b.hits) AS team_hits,
    SUM(b.stat_at_bats) AS team_at_bats
  FROM gold.fact_batting_stats b
  JOIN gold.dim_teams t ON b.team_id = t.team_id
  GROUP BY b.team_id, t.team_name, b.game_date
),

rolling_avgs AS (
  SELECT
    team_id, team_name, game_date,
    ROUND(
      SUM(team_hits) OVER (
        PARTITION BY team_id ORDER BY game_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
      ) / NULLIF(
        SUM(team_at_bats) OVER (
          PARTITION BY team_id ORDER BY game_date
          ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 0
      ), 3
    ) AS rolling_7game_avg,
    ROW_NUMBER() OVER (
      PARTITION BY team_id ORDER BY game_date DESC
    ) AS rn
  FROM team_game_stats
)

SELECT team_id, team_name, game_date, rolling_7game_avg
FROM rolling_avgs
WHERE rn = 1
ORDER BY rolling_7game_avg DESC